In [3]:
!pip install transformers
!pip install "datasets<2.19"
!pip install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


ERROR: Operation cancelled by user
^C


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from tqdm import tqdm
import pickle
import os


MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
LANGUAGES = ["en_us", "de_de", "es_419"]  
MAX_SAMPLES = 30  
OUTPUT_DIR = "./extracted_features"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



print(f"Loading model: {MODEL_NAME}")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()


print(f"Model loaded on {DEVICE}")
print(f"Number of layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")


def extract_representations(audio_array, sampling_rate):
    """Extract hidden states from all layers (XLSR-53 has 24 transformer layers)"""

    # Preprocess audio
    inputs = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(
            inputs["input_values"],
            output_hidden_states=True
        )


    hidden_states = [h.cpu().numpy() for h in outputs.hidden_states]

    return hidden_states

os.makedirs(OUTPUT_DIR, exist_ok=True)

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Processing language: {lang}")
    print(f"{'='*60}")

    # Load dataset
    print(f"Loading FLEURS dataset...")
    dataset = load_dataset(
        "google/fleurs",
        lang,
        split="train",
        trust_remote_code=True
    )

    # Limit samples
    if len(dataset) > MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))

    print(f"Processing {len(dataset)} samples...")

    all_features = []

    for idx, sample in enumerate(tqdm(dataset)):
        try:
            audio = sample['audio']['array']
            sampling_rate = sample['audio']['sampling_rate']

            hidden_states = extract_representations(audio, sampling_rate)

            feature_dict = {
                'id': sample['id'],
                'language': lang,
                'transcription': sample['transcription'],
                'raw_transcription': sample.get('raw_transcription', ''),
                'hidden_states': hidden_states,
                'audio_length': len(audio) / sampling_rate
            }

            all_features.append(feature_dict)

        except Exception as e:
            print(f"\nError on sample {idx}: {e}")
            continue

    output_file = f"{OUTPUT_DIR}/{lang}_features.pkl"
    with open(output_file, 'wb') as f:
        pickle.dump(all_features, f)

    print(f"\nSaved {len(all_features)} samples to {output_file}")

    if len(all_features) > 0:
        sample = all_features[0]
        print(f"\nSample structure:")
        print(f"  Transcription: {sample['transcription'][:50]}...")
        print(f"  Audio length: {sample['audio_length']:.2f}s")
        print(f"  Layer shapes (Total {len(sample['hidden_states'])} layers):")
        for i, h in enumerate(sample['hidden_states']):
            print(f"    Layer {i}: {h.shape}")


print("Done! Features saved to:", OUTPUT_DIR)

Loading model: facebook/wav2vec2-xls-r-300m


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cuda
Number of layers: 24
Hidden size: 1024

Processing language: en_us
Loading FLEURS dataset...


...

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Processing 30 samples...


100%|██████████| 30/30 [00:05<00:00,  5.21it/s]



Saved 30 samples to ./extracted_features/es_419_features.pkl

Sample structure:
  Transcription: los murales o garabatos indeseados reciben el nomb...
  Audio length: 5.76s
  Layer shapes (Total 25 layers):
    Layer 0: (1, 287, 1024)
    Layer 1: (1, 287, 1024)
    Layer 2: (1, 287, 1024)
    Layer 3: (1, 287, 1024)
    Layer 4: (1, 287, 1024)
    Layer 5: (1, 287, 1024)
    Layer 6: (1, 287, 1024)
    Layer 7: (1, 287, 1024)
    Layer 8: (1, 287, 1024)
    Layer 9: (1, 287, 1024)
    Layer 10: (1, 287, 1024)
    Layer 11: (1, 287, 1024)
    Layer 12: (1, 287, 1024)
    Layer 13: (1, 287, 1024)
    Layer 14: (1, 287, 1024)
    Layer 15: (1, 287, 1024)
    Layer 16: (1, 287, 1024)
    Layer 17: (1, 287, 1024)
    Layer 18: (1, 287, 1024)
    Layer 19: (1, 287, 1024)
    Layer 20: (1, 287, 1024)
    Layer 21: (1, 287, 1024)
    Layer 22: (1, 287, 1024)
    Layer 23: (1, 287, 1024)
    Layer 24: (1, 287, 1024)
Done! Features saved to: ./extracted_features


In [2]:
import numpy as np
import pickle
import re
from collections import defaultdict, Counter

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle


FEATURES_DIR = "./extracted_features"
LANGUAGES = ["en_us", "de_de", "es_419"]


def simple_g2p(text):


    text = text.lower().strip()
    text = re.sub(r"[^a-zäöüßàéèêíóúâêîôûçñ ]", "", text)

    consonants = set("bcdfghjklmnpqrstvwxyz")
    vowels = set("aeiou")

    phonemes = []

    for ch in text:
        if ch in consonants:
            phonemes.append(ch)
        elif ch in vowels:
            phonemes.append(ch.upper())

    return phonemes



PHONEME_FEATURES = {
    'p': {'voicing': 'voiceless', 'manner': 'stop'},
    'b': {'voicing': 'voiced', 'manner': 'stop'},
    't': {'voicing': 'voiceless', 'manner': 'stop'},
    'd': {'voicing': 'voiced', 'manner': 'stop'},
    'k': {'voicing': 'voiceless', 'manner': 'stop'},
    'g': {'voicing': 'voiced', 'manner': 'stop'},

    'f': {'voicing': 'voiceless', 'manner': 'fricative'},
    'v': {'voicing': 'voiced', 'manner': 'fricative'},
    's': {'voicing': 'voiceless', 'manner': 'fricative'},
    'z': {'voicing': 'voiced', 'manner': 'fricative'},
    'h': {'voicing': 'voiceless', 'manner': 'fricative'},

    'm': {'voicing': 'voiced', 'manner': 'nasal'},
    'n': {'voicing': 'voiced', 'manner': 'nasal'},

    'l': {'voicing': 'voiced', 'manner': 'liquid'},
    'r': {'voicing': 'voiced', 'manner': 'liquid'},

    'w': {'voicing': 'voiced', 'manner': 'glide'},
    'j': {'voicing': 'voiced', 'manner': 'glide'},
}

def extract_phonological_features(text):
    phonemes = simple_g2p(text)

    if not phonemes:
        return None

    all_feats = defaultdict(list)

    for p in phonemes:
        if p in PHONEME_FEATURES:
            for k, v in PHONEME_FEATURES[p].items():
                all_feats[k].append(v)

    if not all_feats:
        return None

    return {
        feat: Counter(vals).most_common(1)[0][0]
        for feat, vals in all_feats.items()
    }




def load_features(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

def build_dataset(features, layer_idx=6):
    X, y_dict = [], defaultdict(list)

    for sample in features:
        hidden = sample["hidden_states"][layer_idx]

        # (T, D) → mean pooling
        emb = hidden.mean(axis=1).squeeze()

        text = sample.get("transcription", "")

        feats = extract_phonological_features(text)

        if feats is None:
            continue

        X.append(emb)

        for k, v in feats.items():
            y_dict[k].append(v)

    X = np.array(X)

    return X, y_dict


def train_probe(X_train, y_train):
    if len(set(y_train)) < 2:
        return None

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    return model



def cross_lingual(layer=6):

    train_data = load_features(f"{FEATURES_DIR}/en_us_features.pkl")
    X_train, y_train_dict = build_dataset(train_data, layer)

    results = []

    for feat_name, y_train in y_train_dict.items():

        probe = train_probe(X_train, y_train)
        if probe is None:
            continue

        for lang in ["de_de", "es_419"]:
            test_data = load_features(f"{FEATURES_DIR}/{lang}_features.pkl")
            X_test, y_test_dict = build_dataset(test_data, layer)

            if feat_name not in y_test_dict:
                continue

            y_test = y_test_dict[feat_name]

            min_len = min(len(X_test), len(y_test))
            acc = accuracy_score(
                y_test[:min_len],
                probe.predict(X_test[:min_len])
            )

            results.append([feat_name, lang, acc])

            print(f"{feat_name} → {lang}: {acc:.3f}")

    return results


cross_results = cross_lingual(layer=6)




def layer_analysis(layers=[0, 3, 6, 9, 12]):

    data = load_features(f"{FEATURES_DIR}/en_us_features.pkl")

    results = []

    for layer in layers:
        print(f"\nLayer {layer}")

        X, y_dict = build_dataset(data, layer)

        for feat_name, y in y_dict.items():

            split = int(0.8 * len(X))

            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            probe = train_probe(X_train, y_train)

            if probe:
                acc = accuracy_score(y_test, probe.predict(X_test))
                results.append([layer, feat_name, acc])
                print(f"{feat_name}: {acc:.3f}")

    return results


layer_results = layer_analysis()





def language_comparison(layer=6):

    results = []

    for lang in LANGUAGES:

        data = load_features(f"{FEATURES_DIR}/{lang}_features.pkl")
        X, y_dict = build_dataset(data, layer)

        for feat_name, y in y_dict.items():

            split = int(0.8 * len(X))

            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            probe = train_probe(X_train, y_train)

            if probe:
                acc = accuracy_score(y_test, probe.predict(X_test))
                results.append([lang, feat_name, acc])

                print(f"{lang} {feat_name}: {acc:.3f}")

    return results



lang_results = language_comparison(layer=6)



all_results = {
    "cross_lingual": cross_results,
    "layer_analysis": layer_results,
    "language_comparison": lang_results
}

with open(f"{FEATURES_DIR}/results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("Results saved to:", FEATURES_DIR)




voicing → de_de: 0.767
voicing → es_419: 0.833
manner → de_de: 0.700
manner → es_419: 0.567

Layer 0
voicing: 0.500
manner: 0.833

Layer 3
voicing: 0.500
manner: 1.000

Layer 6
voicing: 0.500
manner: 1.000

Layer 9
voicing: 0.500
manner: 0.833

Layer 12
voicing: 0.500
manner: 0.833
en_us voicing: 0.500
en_us manner: 1.000
de_de voicing: 1.000
de_de manner: 0.667
es_419 manner: 0.500
Results saved to: ./extracted_features
